## 1. Configuration file

The original implementation already allowed loading hyperparameters from a YAML file. The configuration file was adapted to the new project.

In [ ]:
# Original
parser = argparse.ArgumentParser(
    description="Learning Hodgkin-Huxley model with DeepONet"
)

parser.add_argument(
    "--config_file",
    type=str,
    default="default_params_don.yml"
)


# Modified
parser = argparse.ArgumentParser(
    description="Burst Classification using DeepONet"
)

parser.add_argument(
    "--config_file",
    type=str,
    default="default_params.yml"
)


### Changes introduced

- Updated the project description.
- Adapted the default configuration file to the neuronal burst classification project.

## 2. Dataset loading

The Hodgkin–Huxley datasets were replaced by the simulated neuronal burst datasets generated from the four-dimensional ODE model.

In [ ]:
# Original
dataset_train = config.get("dataset_train")
dataset_test  = config.get("dataset_test")

u_train, x_train, v_train, scale_fac, _ = load_train(...)
u_test, x_test, v_test, indices = load_test(...)



# Modifies
dataset_train = config.get("dataset_train")
dataset_test  = config.get("dataset_test")

u_train, x_train, v_train, scale_fac, labels = load_train(
    dataset_train,
    scaling,
    labels,
    full_v_data,
    shuffle=True
)

u_test, x_test, v_test, indices = load_test(
    dataset_test,
    scale_fac,
    scaling,
    labels,
    full_v_data,
    shuffle=True
)

### Changes introduced

- Replaced the original Hodgkin–Huxley datasets.
- Training now uses the simulated burst datasets generated during this project.
- Included burst labels for future classification tasks.

## 3. DeepONet architecture

The DeepONet architecture was adapted to receive four physical parameters instead of the original inputs.

In [ ]:
#Original
layers = {
    "branch": [u_dim] + inner_layer_b + [G_dim],
    "trunk": [x_dim*(N_FourierF==0)+2*N_FourierF]
              + inner_layer_t
              + [G_dim]
}

#Modified
layers = {
    "branch": [4] + inner_layer_b + [128],
    "trunk": [1] + inner_layer_t + [128]
}

### Changes introduced

- Increased the Branch Network input dimension to four physical parameters.
- Reduced the latent representation size.
- Simplified the architecture to improve computational efficiency.

## 4. DataLoader

The DataLoader was adapted to the new burst dataset.

In [ ]:
#Original
train_loader = DataLoader(
    TensorDataset(v_train, u_train),
    batch_size=batch_size,
    shuffle=True
)

#Modified
train_loader = DataLoader(
    TensorDataset(v_train, u_train),
    batch_size=batch_size,
    shuffle=True,
    generator=torch.Generator(device=mydevice)
)

### Changes introduced

- Preserved random shuffling.
- Added a device-specific random generator for improved GPU compatibility.


## 5. Model selection

The repository originally supported several neural operator architectures. During this project, the focus was placed on DeepONet while preparing the infrastructure for future PINN integration.

In [ ]:
#Original
if arc=="DON":
    ...
elif arc=="DON_md":
    ...
elif arc=="WNO":
    ...
elif arc=="FNO":
    ...
elif arc=="AdFNO":
    ...


#Modified
if arc == "DON":
    model = DeepONet(...)

### Changes introduced

- The current implementation focuses on DeepONet.
- PINN support is currently under development through the new `physics.py` module.
- Other neural operators remain available for future comparisons.

## 6. Training

The training workflow remained largely unchanged, but it now operates on the newly generated neuronal burst datasets.

In [ ]:
#Original
trainer = Training(...)

trainer.train()

torch.save(model, name_model)

#Modified
trainer = Training(...)

history = trainer.train()

torch.save(model, name_model)

### Changes introduced

- The training history is now returned to allow visualization of the learning curves.
- The saved model is compatible with the updated dataset and DeepONet configuration.